# AGEMR B1：线性 SNP × age interaction（1 df）

**目标：** 使用现有 All of Us / Verily Workbench 持久文件，分析 BMI、SBP、DBP。保留原来的完整非线性年龄样条、sex-at-birth 和 QPC 调整空间，只检验一个中心化线性基因×年龄项。

本 notebook 根据上传的 `age_varying(1).ipynb` 编写。原 notebook 的记录为 PLINK **v2.0.0-a.7.8（19 Sep 2026）**；原路径、表型处理、19 个协变量与 `BMI_chr1`…`BMI_chr22` 共用基因型面板均已逐项读取。这里不重跑 cohort 构建、不重新选 SNP、不查询 All of Us 云端数据、不下载 controlled data。若当前运行环境缺少 PLINK2，会从公开 Bioconda 安装原版软件到 workspace 持久目录；不安装到当前 Python/Spark 环境。新文件全部留在同一 workspace 持久目录。

**运行：** 在现有 JupyterLab Spark 环境选择支持本机 Python 的 kernel，确认下一节路径后 Run All。PLINK 在 driver 上按 chromosome 顺序运行；不使用 Spark executors，不需要已有 Python 变量或 Spark session。需要已安装的 numpy、pandas、scipy、matplotlib、patsy。先自动完成合成数据验证，再批量分析真实文件；任何核验失败都会停止。

交付版本没有真实数据执行输出。原 notebook 中的旧结果属于历史记录；本 notebook 将读取 workspace 的真实结果文件重新汇总，不把它们当作新线性分析结果。


## 1. 模型与参数选择

令 $A_c=(A-\bar A)/10$，每个 phenotype 的 $\bar A$ 取其原 B1 完整分析样本的平均年龄，之后在所有 SNP 上固定：

$$Y_i=\alpha+\beta_GG_i+\theta G_iA_{c,i}+b(A_i)^T\delta+C_i^T\xi+\epsilon_i.$$

检验 $H_0:\theta=0$，是 **1-df interaction-only 检验**。$\theta$ 的单位为“每个 A1 allele、每 10 年的 phenotype effect 变化”；`ADD` 是中心年龄处的效应，不是 pooled GWAS effect。

原 natural cubic spline 的 `B1…B4` 连同截距已经包含线性年龄。为避免重复加入 age 导致完全共线，代码将它们等价改写成 `AGE10 + NL1 + NL2 + NL3`：先从原四列 spline 投影掉截距和线性年龄，取剩余三个独立方向。代码数值验证前后列空间相同；不会把 nuisance age 效应限制为线性。原 sex / QPC 列按原值保留。

协变量文件顺序固定为 `AGE10, NL1, NL2, NL3, 原 sex 列, 原 QPC 列`。若共 $q=19$ 个协变量：

| 原始编号（不含截距） | 项 | 保留？ |
|---|---|---|
| 1 | ADD | 是 |
| 2–20 | 所有协变量主效应 | 是 |
| 21 | ADD×AGE10 | 是，唯一被检验项 |
| 22–39 | 其他交互项 | 否 |

因此使用 `--glm interaction --parameters 1-21 --tests 21`，编号由实际协变量数计算。保留 covariate 输出，以明确读取 `ADDxAGE10` 的 BETA/SE；同时核验 `USER_1DF` 的 $F=t^2$；主分析使用 `ADDxAGE10` 的双侧 t-test P。`--tests` 不会自行删除其他交互项，必须配合 `--parameters`。不启用 `hide-covar`、`qt-residualize`、`omit-ref` 或额外的 `sex` modifier。

参数依据：[PLINK association 文档](https://www.cog-genomics.org/plink/2.0/assoc#glm)、目标可执行程序的 `--help` 和本 notebook 的独立 OLS 合成数据检验。核查日期：2026-10-28。


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, re, uuid, warnings
from datetime import datetime, timezone
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from patsy import dmatrix
from IPython.display import display

# 原 notebook 的真实持久路径；仅在 workspace 移动过时修改。
FINAL = Path.home() / "workspace" / "agemr-data" / "final_cohort_v1"
B1 = FINAL / "B1_age_varying"
GENO = FINAL / "genotypes_White_EUR"
TRAITS = ("BMI", "SBP", "DBP")
CHROMS = tuple(range(1, 23))
SOURCE = {"BMI": FINAL / "bmi_primary_earliest.tsv",
          "SBP": FINAL / "bp_primary_earliest.tsv",
          "DBP": FINAL / "bp_primary_earliest.tsv"}
PLINK2 = None  # 可填写已有可执行文件的绝对路径；None 表示自动查找。
INSTALL_PLINK_IF_MISSING = True  # 只下载公开软件包；设为 False 可禁用联网安装。
PLINK_ENV = FINAL.parent / "tools" / "plink2_2.0.0a.7.8"
THREADS = 2
MEMORY_MB = 1024  # 与旧 driver 约 8 GB RAM 相容；单进程、顺序运行。
ALPHA = 0.05
EXPECTED_PANEL_N = 1902  # 仅用于核对旧面板，不作为实际多重检验分母。
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN = B1 / "B1_linear_age" / RUN_ID
FIG = RUN / "figures"
QC = RUN / "qc"
for folder in (FIG, QC, RUN / "inputs", RUN / "synthetic"):
    folder.mkdir(parents=True, exist_ok=True)
print("New run directory:", RUN)


## 2. 软件准备、文件与版本核验

**修订说明（PLINK2 not found）：** 上一版只检查原 `~/plink2_env/bin/plink2`，未处理该环境已不存在的情况。现在先试用现有程序并运行 `--version`，找不到时使用现有 mamba/conda，把 `plink2=2.0.0a.7.8` 安装到 `~/workspace/agemr-data/tools/plink2_2.0.0a.7.8`。版本名已根据 [Bioconda 官方 recipe](https://bioconda.github.io/recipes/plink2/README.html) 核对。

这一步需要 Workbench 能连接公开软件仓库。安装失败会保存日志并停止，不会开始真实数据分析或删除已有目录。安装成功后重复运行会复用该程序，无需激活 conda 环境。

必需输入：三个表型各自的 `.pheno.tsv`、`.covar.tsv`、`.meta.json`；两个原始 phenotype 表；22 套 `BMI_chr*.pgen/.pvar/.psam`；三个旧 `*_age_varying_joint_tests.tsv`。不从旧 notebook 的内存对象获取数据。

仅接受这里已知的 IID-only / FID=0 结构。重复 IID、重复 variant ID、多等位 variant、不同染色体 sample 列表不一致、文件缺失均明确报错。基因型大文件记录路径、大小和修改时间；小表记录 SHA256。不会将 controlled inputs 复制出 workspace。


In [ ]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str) + "\n")

def read_table(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)

def indexed_iid(df, label):
    x = df.rename(columns={"#IID": "IID", "#FID": "FID"}).copy()
    require("IID" in x, f"{label}: IID missing")
    require(x["IID"].ne("").all() and x["IID"].is_unique, f"{label}: empty/duplicate IID")
    if "FID" in x:
        require(x["FID"].eq("0").all(), f"{label}: nonzero FID; explicit ID mapping required")
        x = x.drop(columns="FID")
    return x.set_index("IID", verify_integrity=True)

def numeric(df, label):
    x = df.apply(pd.to_numeric, errors="raise").astype(float)
    require(np.isfinite(x.to_numpy()).all(), f"{label}: missing/NaN/Inf values")
    return x

def checked_run(cmd, prefix):
    prefix = Path(prefix)
    prefix.parent.mkdir(parents=True, exist_ok=True)
    save_json(str(prefix) + ".command.json", cmd)
    with open(str(prefix) + ".stdout.txt", "w") as out, open(str(prefix) + ".stderr.txt", "w") as err:
        result = subprocess.run([str(v) for v in cmd], stdout=out, stderr=err, check=False)
    require(result.returncode == 0,
            f"PLINK failed ({result.returncode}); inspect {prefix}.stderr.txt and {prefix}.log")

def resolve_plink2(preferred, install_if_missing, env_prefix, log_dir):
    # Only inspect known software locations; do not traverse genotype data.
    candidates = [preferred, str(env_prefix / "bin" / "plink2"), shutil.which("plink2"),
                  str(Path.home() / "plink2_env" / "bin" / "plink2"),
                  str(Path(sys.prefix) / "bin" / "plink2"),
                  "/opt/conda/default/bin/plink2", "/opt/conda/bin/plink2",
                  "/usr/local/bin/plink2", "/usr/bin/plink2"]
    audit = []
    def probe(candidate):
        if not candidate:
            return None
        path = Path(candidate).expanduser()
        if not path.is_file() or not os.access(path, os.X_OK):
            audit.append({"path": str(path), "status": "missing_or_not_executable"})
            return None
        try:
            result = subprocess.run([str(path), "--version"], capture_output=True, text=True, timeout=30)
            version = (result.stdout + result.stderr).strip()
            good = result.returncode == 0 and re.search(r"PLINK\s+v?2\.", version, re.I)
            audit.append({"path": str(path), "status": "usable" if good else "version_check_failed",
                          "version_output": version})
            return str(path) if good else None
        except (OSError, subprocess.TimeoutExpired) as exc:
            audit.append({"path": str(path), "status": "execution_failed", "error": str(exc)})
            return None
    for candidate in dict.fromkeys(candidates):
        found = probe(candidate)
        if found:
            save_json(log_dir / "plink_discovery.json", audit)
            return found
    save_json(log_dir / "plink_discovery.json", audit)
    require(install_if_missing,
            f"No usable PLINK2 found. See {log_dir / 'plink_discovery.json'}. "
            "Set PLINK2 to an installed executable or enable INSTALL_PLINK_IF_MISSING.")
    managers = [shutil.which("mamba"), "/opt/conda/default/bin/mamba",
                shutil.which("conda"), "/opt/conda/default/bin/conda", "/opt/conda/bin/conda"]
    manager = next((p for p in managers if p and Path(p).is_file() and os.access(p, os.X_OK)), None)
    require(manager is not None,
            "No mamba/conda found. Set PLINK2 to an existing installed program, "
            "or install public plink2=2.0.0a.7.8 using your Workbench software setup.")
    # Never alter the active Python/Spark environment or delete a partial prefix.
    if env_prefix.exists():
        raise RuntimeError(f"No usable PLINK2 in existing destination {env_prefix}. "
                           "Inspect it or choose a new PLINK_ENV directory; it will not be overwritten.")
    env_prefix.parent.mkdir(parents=True, exist_ok=True)
    command = [manager, "create", "-y", "-p", str(env_prefix), "--override-channels",
               "-c", "conda-forge", "-c", "bioconda", "plink2=2.0.0a.7.8"]
    save_json(log_dir / "plink_install.command.json", command)
    print("PLINK2 is missing. Installing public software only into:", env_prefix, flush=True)
    print("The package manager output below shows installation progress.", flush=True)
    with (log_dir / "plink_install.log").open("w") as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = proc.wait()
        except BaseException:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            raise
    require(returncode == 0,
            f"Public-software installation failed. See {log_dir / 'plink_install.log'}. "
            "Analysis has not started. If interrupted, inspect the partial environment or choose a new PLINK_ENV.")
    found = probe(str(env_prefix / "bin" / "plink2"))
    save_json(log_dir / "plink_discovery.json", audit)
    require(found is not None, "Installation finished but PLINK2 cannot execute; inspect installation and discovery logs.")
    return found

PLINK2 = resolve_plink2(PLINK2, INSTALL_PLINK_IF_MISSING, PLINK_ENV, QC)
print("Using PLINK2:", PLINK2)
version_result = subprocess.run([PLINK2, "--version"], capture_output=True, text=True, check=True)
PLINK_VERSION = version_result.stdout.strip()
print(PLINK_VERSION)
help_result = subprocess.run([PLINK2, "--help", "glm", "parameters", "tests", "no-input-missing-phenotype"],
                             capture_output=True, text=True, check=True)
(QC / "plink_help.txt").write_text(help_result.stdout)
(QC / "plink_version.txt").write_text(PLINK_VERSION + "\n")

required = list(dict.fromkeys(SOURCE.values()))
for trait in TRAITS:
    required += [B1 / f"{trait}.{ext}" for ext in ("pheno.tsv", "covar.tsv", "meta.json")]
    required += [B1 / f"{trait}_age_varying_joint_tests.tsv"]
for c in CHROMS:
    required += [GENO / f"BMI_chr{c}.{ext}" for ext in ("pgen", "pvar", "psam")]
missing = [str(p) for p in required if not p.is_file()]
require(not missing, "Missing existing workspace inputs:\n" + "\n".join(missing))
file_records = []
for p in required:
    st = p.stat()
    file_records.append({"path": str(p), "bytes": st.st_size, "mtime_ns": st.st_mtime_ns,
                         "sha256": None if p.suffix == ".pgen" else sha256(p)})
save_json(QC / "input_files.json", file_records)
save_json(QC / "environment.json", {
    "python": sys.version, "plink": PLINK_VERSION, "threads": THREADS, "memory_mb": MEMORY_MB,
    "packages": {m: importlib.metadata.version(m) for m in ("numpy", "pandas", "scipy", "matplotlib", "patsy")},
    "run": str(RUN), "created_utc": datetime.now(timezone.utc).isoformat()})

panels = []
genotype_ids = None
for c in CHROMS:
    psam = indexed_iid(pd.read_csv(GENO / f"BMI_chr{c}.psam", sep=r"\s+", dtype=str,
                                   keep_default_na=False), f"chr{c} psam")
    if genotype_ids is None:
        genotype_ids = psam.index
    else:
        require(psam.index.equals(genotype_ids), f"chr{c}: genotype sample order/set differs")
    pvar_path = GENO / f"BMI_chr{c}.pvar"
    with pvar_path.open() as f:
        skip = 0
        for line in f:
            if line.startswith("##"):
                skip += 1
            else:
                break
    pv = pd.read_csv(pvar_path, sep=r"\s+", skiprows=skip, dtype=str, keep_default_na=False)
    require(set(["#CHROM", "POS", "ID", "REF", "ALT"]).issubset(pv), f"{pvar_path}: unexpected columns")
    pv = pv[["#CHROM", "POS", "ID", "REF", "ALT"]].copy()
    pv["#CHROM"] = pv["#CHROM"].str.replace(r"^chr", "", regex=True)
    require(pv["#CHROM"].eq(str(c)).all() and len(pv) > 0, f"{pvar_path}: chromosome mismatch/empty")
    require(not pv["ALT"].str.contains(",", regex=False).any(), "Multiallelic variant: parameter count would change")
    require(pv["ID"].ne(".").all() and pv["ID"].ne("").all(), "Missing variant ID")
    pv["POS"] = pd.to_numeric(pv["POS"], errors="raise").astype(int)
    panels.append(pv)
PANEL = pd.concat(panels, ignore_index=True)
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
require(PANEL["ID"].is_unique and not PANEL.duplicated(KEY).any(), "Duplicate variant key/ID")
PANEL.to_csv(QC / "variant_panel.tsv", sep="\t", index=False)
M_PANEL = len(PANEL)
if M_PANEL != EXPECTED_PANEL_N:
    warnings.warn(f"Current panel={M_PANEL}; original notebook recorded {EXPECTED_PANEL_N}. Inspect before interpretation.")
print("Genotype samples:", len(genotype_ids), "Variants:", M_PANEL)


## 3. 从保存的 B1 inputs 构造等价 nuisance 设计

不重新筛样本、不重算结点或 QPC。年龄只从与旧 phenotype 相同的 primary-earliest 表中按 IID 合并，并核对 phenotype 数值。旧 meta 给出年龄范围和结点；要求现有样本全部符合它。输出前核验矩阵秩、前后列空间误差、VIF 与最大相关系数。

`AGE10` 保持“每 10 年”的尺度，不再进行协变量统一方差标准化。所有新输入均无缺失值，使用 `--no-input-missing-phenotype`，以免合法的 -9 被当作缺失；这也修正了旧日志中出现的 -9 警告。若旧协变量恰有 -9，比较表会特别标记历史样本数变化的可能性。[PLINK missing-value 定义](https://www.cog-genomics.org/plink/2.0/input#pheno)


In [ ]:
def rebase_spline(B, age):
    B = np.asarray(B, dtype=float)
    age = np.asarray(age, dtype=float)
    n = len(age)
    age_mean = float(age.mean())
    a = (age - age_mean) / 10.0
    Z = np.column_stack([np.ones(n), a])
    old = np.column_stack([np.ones(n), B])
    fit_age = np.linalg.lstsq(old, a, rcond=None)[0]
    age_error = float(np.max(np.abs(a - old @ fit_age)))
    require(age_error < 1e-7, f"Age not in saved spline span: max error {age_error}")
    projection = np.linalg.lstsq(Z, B, rcond=None)[0]
    R = B - Z @ projection
    U, s, Vt = np.linalg.svd(R, full_matrices=False)
    rank = int(np.sum(s > s[0] * 1e-9))
    require(rank == 3, f"Expected 3 nonlinear directions, got {rank}")
    T = Vt[:3].T * (np.sqrt(n - 1) / s[:3])
    NL = R @ T
    new = np.column_stack([Z, NL])
    reconstruction = np.linalg.lstsq(new, old, rcond=None)[0]
    span_error = float(np.max(np.abs(old - new @ reconstruction)))
    require(span_error < 1e-7 and np.linalg.matrix_rank(new) == 5,
            f"Spline-space equivalence failed: {span_error}")
    return np.column_stack([a, NL]), {
        "age_center_years": age_mean, "age_unit_years": 10,
        "age_span_max_abs_error": age_error, "spline_span_max_abs_error": span_error,
        "nonlinear_rank": rank, "residual_singular_values": s.tolist(),
        "B_projection_on_intercept_AGE10": projection.tolist(),
        "residual_B_to_NL": T.tolist(), "old_from_new": reconstruction.tolist()}

def design_qc(X):
    A = X.to_numpy(dtype=float)
    full = np.column_stack([np.ones(len(A)), A])
    require(np.linalg.matrix_rank(full) == full.shape[1], "Rank-deficient nuisance design")
    require((A.std(axis=0, ddof=1) > 0).all(), "Constant covariate")
    corr = np.corrcoef(A, rowvar=False)
    vif = np.diag(np.linalg.inv(corr))
    maxcorr = float(np.max(np.abs(corr - np.eye(len(corr)))))
    require(maxcorr < 0.999 and float(vif.max()) < 50,
            "Nuisance multicollinearity exceeds default PLINK limits; investigate, do not relax blindly")
    return pd.DataFrame({"covariate": X.columns, "VIF_with_intercept": vif,
                         "mean": A.mean(axis=0), "sd": A.std(axis=0, ddof=1)}), maxcorr

prepared = {}
cohort_qc = []
for trait in TRAITS:
    ph = indexed_iid(read_table(B1 / f"{trait}.pheno.tsv"), f"{trait} pheno")
    cv = indexed_iid(read_table(B1 / f"{trait}.covar.tsv"), f"{trait} covar")
    require(set(ph.index) == set(cv.index), f"{trait}: saved pheno/covar IID sets differ")
    cv = numeric(cv.loc[ph.index], f"{trait} covar")
    y = numeric(ph[[trait]], f"{trait} pheno")
    require((y[trait] > 0).all(), f"{trait}: unexpected nonpositive primary phenotype")
    require(ph.index.isin(genotype_ids).all(), f"{trait}: saved sample absent from genotype panel")
    meta = json.loads((B1 / f"{trait}.meta.json").read_text())
    require(meta["N"] == len(y) and meta["K_spline"] == 4, f"{trait}: metadata mismatch")
    require(list(cv.columns) == meta["covariate_columns"], f"{trait}: covariate order differs from metadata")
    require(list(cv.columns[:4]) == ["B1", "B2", "B3", "B4"], f"{trait}: original spline columns missing")
    src = indexed_iid(pd.read_csv(SOURCE[trait], sep="\t", usecols=["IID", "age", trait],
                                  dtype={"IID": str}), f"{trait} source")
    require(ph.index.isin(src.index).all(), f"{trait}: age missing from source")
    src = numeric(src.loc[ph.index, ["age", trait]], f"{trait} matched source")
    require(np.allclose(y[trait], src[trait], rtol=1e-10, atol=1e-10), f"{trait}: source phenotype changed")
    age = src["age"]
    require(age.between(meta["age_lo"], meta["age_hi"]).all(), f"{trait}: age outside original support")
    A, transform = rebase_spline(cv.iloc[:, :4], age)
    nuisance = pd.DataFrame(A, index=cv.index, columns=["AGE10", "NL1", "NL2", "NL3"])
    nuisance = pd.concat([nuisance, cv.iloc[:, 4:]], axis=1)
    vif, maxcorr = design_qc(nuisance)
    vif.to_csv(QC / f"{trait}_nuisance_QC.tsv", sep="\t", index=False)
    q = nuisance.shape[1]
    parameter_table = pd.DataFrame({"index": range(1, 2 * q + 2),
        "term": ["ADD"] + nuisance.columns.tolist() + ["ADDx" + c for c in nuisance.columns]})
    parameter_table["included"] = parameter_table["index"] <= q + 2
    parameter_table["tested"] = parameter_table["index"] == q + 2
    parameter_table.to_csv(QC / f"{trait}_parameters.tsv", sep="\t", index=False)
    pheno_path = RUN / "inputs" / f"{trait}.pheno.tsv"
    covar_path = RUN / "inputs" / f"{trait}.covar.tsv"
    y.to_csv(pheno_path, sep="\t", index_label="#IID", float_format="%.17g", na_rep="NA")
    nuisance.to_csv(covar_path, sep="\t", index_label="#IID", float_format="%.17g", na_rep="NA")
    transform.update({"trait": trait, "N": len(y), "covariates": nuisance.columns.tolist(),
                      "original_meta": meta, "exact_neg9_original_covariates": int((cv == -9).sum().sum()),
                      "pheno_sha256": sha256(pheno_path), "covar_sha256": sha256(covar_path)})
    save_json(RUN / "inputs" / f"{trait}.meta.json", transform)
    prepared[trait] = {"pheno": pheno_path, "covar": covar_path, "q": q,
                       "N": len(y), "transform": transform}
    cohort_qc.append({"trait": trait, "N": len(y), "q": q, "age_min": age.min(),
                      "age_max": age.max(), "age_center": transform["age_center_years"],
                      "span_error": transform["spline_span_max_abs_error"], "max_covariate_corr": maxcorr,
                      "max_VIF": vif["VIF_with_intercept"].max(),
                      "old_exact_neg9": transform["exact_neg9_original_covariates"]})
    del cv, ph, src, y, nuisance, A
cohort_qc = pd.DataFrame(cohort_qc)
cohort_qc.to_csv(QC / "cohorts.tsv", sep="\t", index=False)
display(cohort_qc)


## 4. PLINK 命令、输出审计和独立合成数据验证

每次真实运行之前，用固定种子的独立合成数据（含缺失 genotype）检查：交互项名称与数量、BETA/SE、双侧 t 检验、1-df F 统计量及其自由度约定、sample count，以及原 spline 与重参数化设计所得交互系数的一致性。此处的模拟结果仅用于验证程序，不能用于推断 All of Us。

**实测版本差异：** 同日期 a.7.8 macOS M1 可执行程序的合成测试中，`USER_1DF` 的统计量等于 $t^2$，但其 P 使用 F(1, OBS_CT)，而单项 `ADDxAGE10` P 使用 t(OBS_CT − 参数总数)。这与官方源代码中 joint-test 调用相符。因此二者 P 不要求完全相等；主分析只采用经独立 OLS 核验的单项 t-test P。代码记录 joint P 的自由度约定，另外保存按正确 residual df 重算的 1-df F P。若目标版本已改变约定，也会识别；其他不相容结果会停止。[官方 GLM 源码](https://github.com/chrchang/plink-ng/blob/master/2.0/plink2_glm_linear.cc)

默认 PLINK OLS 使用常规标准误，不是 heteroskedasticity-robust / mixed model。保留默认共线性检查，遇到 ERRCODE 不提高阈值掩盖问题。


In [ ]:
def glm_command(pfile, pheno, trait, covar, q, out):
    # q covariates; AGE10 is first. No dominance, local covar, condition or extra sex.
    return [PLINK2, "--pfile", str(pfile), "--pheno", str(pheno), "--pheno-name", trait,
            "--covar", str(covar), "--glm", "interaction",
            "--parameters", f"1-{q + 2}", "--tests", str(q + 2),
            "--no-input-missing-phenotype", "--threads", str(THREADS),
            "--memory", str(MEMORY_MB), "--out", str(out)]

def read_glm(path):
    z = read_table(path)
    need = set(KEY + ["A1", "TEST", "OBS_CT", "BETA", "SE", "T_OR_F_STAT", "P", "ERRCODE", "A1_FREQ"])
    require(need.issubset(z.columns), f"Unexpected GLM schema in {path}: {list(z.columns)}")
    z["#CHROM"] = z["#CHROM"].str.replace(r"^chr", "", regex=True)
    for col in ("POS", "OBS_CT", "BETA", "SE", "T_OR_F_STAT", "P", "A1_FREQ"):
        z[col] = pd.to_numeric(z[col], errors="coerce")
    return z

def extract_interaction(z, q, expected, label):
    terms = set(z["TEST"])
    require({"ADD", "ADDxAGE10", "USER_1DF"}.issubset(terms), f"{label}: missing test rows: {terms}")
    require({s for s in terms if s.startswith("ADDx")} == {"ADDxAGE10"},
            f"{label}: unexpected additional interactions")
    key = KEY + ["A1"]
    interaction = z.loc[z["TEST"].eq("ADDxAGE10")].copy()
    joint = z.loc[z["TEST"].eq("USER_1DF")].copy()
    require(not interaction.duplicated(KEY).any() and not joint.duplicated(KEY).any(), f"{label}: duplicate test rows")
    audit = expected[KEY].merge(interaction[KEY], how="outer", on=KEY, indicator=True, validate="one_to_one")
    require(audit["_merge"].eq("both").all(), f"{label}: missing/unexpected variants")
    require(len(joint) == len(interaction), f"{label}: missing joint tests")
    j = joint[key + ["P", "T_OR_F_STAT", "ERRCODE", "OBS_CT"]].rename(columns={
        "P": "P_joint", "T_OR_F_STAT": "F_joint", "ERRCODE": "ERRCODE_joint", "OBS_CT": "N_joint"})
    x = interaction.merge(j, on=key, how="left", validate="one_to_one")
    require(x["ERRCODE_joint"].notna().all(), f"{label}: joint allele/key mismatch")
    # q main covariates + G + GxAGE10 + intercept = q+3 estimated coefficients.
    x["DF_RESID"] = x["OBS_CT"] - (q + 3)
    valid = (x["ERRCODE"].eq(".") & x["ERRCODE_joint"].eq(".") &
             x["P"].between(0, 1) & x["P_joint"].between(0, 1) &
             np.isfinite(x["BETA"]) & np.isfinite(x["SE"]) & x["SE"].gt(0) &
             np.isfinite(x["T_OR_F_STAT"]) & x["DF_RESID"].gt(0))
    x["VALID"] = valid
    v = x.loc[valid]
    require(np.allclose(v["OBS_CT"], v["N_joint"], rtol=0, atol=0), f"{label}: sample counts differ")
    require(np.allclose(v["F_joint"], v["T_OR_F_STAT"] ** 2, rtol=5e-5, atol=1e-8), f"{label}: F != t^2")
    x["P_F_residual_df"] = stats.f.sf(x["F_joint"], 1, x["DF_RESID"])
    x["P_F_OBS_CT_df"] = stats.f.sf(x["F_joint"], 1, x["OBS_CT"])
    resid_match = np.isclose(v["P_joint"], x.loc[valid, "P_F_residual_df"], rtol=5e-4, atol=1e-300)
    obs_match = np.isclose(v["P_joint"], x.loc[valid, "P_F_OBS_CT_df"], rtol=5e-4, atol=1e-300)
    require((resid_match | obs_match).all(), f"{label}: unrecognized joint-test P convention")
    x["joint_P_df_convention"] = "invalid"
    x.loc[valid, "joint_P_df_convention"] = np.where(resid_match & obs_match, "both_within_rounding",
                                                     np.where(resid_match, "residual_df", "OBS_CT_df"))
    require(np.allclose(v["P"], x.loc[valid, "P_F_residual_df"], rtol=5e-4, atol=1e-300),
            f"{label}: single-term P differs from F(1,residual_df)")
    require(np.allclose(v["T_OR_F_STAT"], v["BETA"] / v["SE"], rtol=5e-5, atol=1e-8), f"{label}: t != beta/SE")
    lp = np.log(2) + stats.t.logsf(np.abs(v["T_OR_F_STAT"]), v["DF_RESID"])
    nonzero = v["P"].gt(0)
    require(np.allclose(np.log(v.loc[nonzero, "P"]), lp[nonzero], rtol=3e-4, atol=3e-4),
            f"{label}: reported P inconsistent with residual df")
    x["BETA_per_10y"] = x["BETA"]
    x["SE_per_10y"] = x["SE"]
    x["BETA_per_year"] = x["BETA"] / 10
    x["SE_per_year"] = x["SE"] / 10
    crit = stats.t.ppf(0.975, x["DF_RESID"])
    x["CI95_low_per_10y"] = x["BETA"] - crit * x["SE"]
    x["CI95_high_per_10y"] = x["BETA"] + crit * x["SE"]
    return x

def ols_reference(y, X):
    coef, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
    require(rank == X.shape[1], "Synthetic reference design is singular")
    df = len(y) - rank
    residual = y - X @ coef
    se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (residual @ residual / df))
    t = coef[-1] / se[-1]
    return coef[-1], se[-1], 2 * stats.t.sf(abs(t), df)

def synthetic_validation():
    d = RUN / "synthetic"
    rng = np.random.default_rng(20261028)
    n, m = 1200, 4
    ids = [f"sim{i:05d}" for i in range(n)]
    age = rng.uniform(20, 86, n)
    B = np.asarray(dmatrix("cr(age, knots=(42,58,69), lower_bound=20, upper_bound=86, constraints='center') - 1",
                          {"age": age}))
    rebased, meta = rebase_spline(B, age)
    extra = np.column_stack([rng.binomial(1, 0.48, n), rng.normal(size=(n, 2))])
    extra[0, 1] = -9.0  # A valid numeric covariate, not a missing value.
    X = np.column_stack([rebased, extra])
    G = np.column_stack([rng.binomial(2, f, n) for f in (0.12, 0.23, 0.36, 0.08)]).astype(float)
    a = rebased[:, 0]
    y = 24 + B @ np.array([1.2, -0.7, 0.4, 0.8]) + extra @ np.array([0.7, 0.2, -0.3])
    y += 0.35 * G[:, 0] + 0.22 * G[:, 0] * a - 0.18 * G[:, 1] * a + rng.normal(0, 1.2, n)
    for j in range(m):
        G[rng.choice(n, 11 + j, replace=False), j] = np.nan
    vcf = d / "toy.vcf"
    with vcf.open("w") as f:
        f.write('##fileformat=VCFv4.2\n##contig=<ID=1>\n##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t" + "\t".join(ids) + "\n")
        for j in range(m):
            gt = ["./." if np.isnan(g) else ["0/0", "0/1", "1/1"][int(g)] for g in G[:, j]]
            f.write(f"1\t{1001+j}\tsimvar{j}\tA\tC\t.\tPASS\t.\tGT\t" + "\t".join(gt) + "\n")
    pfile = d / "toy"
    checked_run([PLINK2, "--vcf", str(vcf), "--double-id", "--make-pgen", "--out", str(pfile)], d / "convert_initial")
    # --double-id sets FID=IID; use explicit matching FID/IID in this synthetic-only fixture.
    cov = pd.DataFrame(X, columns=["AGE10", "NL1", "NL2", "NL3", "SEX_DUMMY", "PC1", "PC2"])
    cov.insert(0, "IID", ids); cov.insert(0, "#FID", ids)
    ph = pd.DataFrame({"#FID": ids, "IID": ids, "SIM": y})
    cov.to_csv(d / "cov.tsv", sep="\t", index=False, float_format="%.17g")
    ph.to_csv(d / "pheno.tsv", sep="\t", index=False, float_format="%.17g")
    out = d / "fit"
    q = X.shape[1]
    checked_run(glm_command(pfile, d / "pheno.tsv", "SIM", d / "cov.tsv", q, out), out)
    z = read_glm(str(out) + ".SIM.glm.linear")
    expected = pd.DataFrame({"#CHROM": "1", "POS": np.arange(1001,1001+m),
                             "ID": [f"simvar{j}" for j in range(m)], "REF": "A", "ALT": "C"})
    result = extract_interaction(z, q, expected, "synthetic").set_index("ID")
    require(result["VALID"].all(), "Synthetic PLINK regression failed")
    checks = []
    for j in range(m):
        keep = np.isfinite(G[:, j])
        g = G[keep, j]
        row = result.loc[f"simvar{j}"]
        require(row["A1"] == "C", "Unexpected synthetic A1 orientation")
        design = np.column_stack([np.ones(keep.sum()), g, X[keep], g*a[keep]])
        original = np.column_stack([np.ones(keep.sum()), g, B[keep], extra[keep], g*a[keep]])
        beta, se, p = ols_reference(y[keep], design)
        b0, se0, p0 = ols_reference(y[keep], original)
        require(np.allclose([beta,se,p], [b0,se0,p0], rtol=1e-8, atol=1e-12), "Changed nuisance space")
        require(np.allclose([row["BETA"],row["SE"],row["P"]], [beta,se,p], rtol=5e-5, atol=1e-12),
                "PLINK vs independent OLS mismatch")
        require(row["OBS_CT"] == keep.sum(), "Missing genotype/sample count handling differs")
        checks.append({"variant": f"simvar{j}", "N": int(keep.sum()), "BETA_plink": row["BETA"],
                       "BETA_OLS": beta, "SE_plink": row["SE"], "SE_OLS": se,
                       "P_plink": row["P"], "P_OLS": p, "passed": True})
    checks = pd.DataFrame(checks)
    checks.to_csv(QC / "synthetic_validation.tsv", sep="\t", index=False)
    print("Synthetic validation passed; no All of Us observations used.")
    return checks

synthetic_checks = synthetic_validation()
display(synthetic_checks)


## 5. 批量运行 BMI / SBP / DBP × chr1–22

每个 phenotype 都使用原来的 `BMI_chr*` genotype 面板；文件前缀的 BMI 不表示 SBP/DBP 需要更换 genotype。输出位于本次新建的运行目录，原 B1 文件不会覆盖。每个 chromosome 都检查面板覆盖、检验名称、P 值与自由度；PLINK 返回成功但存在无效 SNP 时，保留错误记录并在 QC 中统计。

全部结果作为预先指定的面板检验族：trait 内 Bonferroni 分母为完整输入面板大小；跨三个 trait 的补充校正使用 3×面板大小。BH 对失败项临时以 P=1 计入分母，但失败项本身的校正结果保留缺失。BH 需要独立或适当的正依赖条件；额外给出 BY 作为任意依赖下更保守的参照。不按先前显著结果筛选本次 SNP。


In [ ]:
def adjusted_p(p, method="BH"):
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranks = np.arange(1, m+1)
    factor = np.sum(1 / ranks) if method == "BY" else 1.0
    vals = np.minimum.accumulate((p[order] * m * factor / ranks)[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.minimum(vals, 1.0)
    return out

def annotate_multiple_testing(df, pcol="P", validcol="VALID"):
    x = df.copy()
    require(len(x) == M_PANEL, "Multiplicity family must contain the entire input panel")
    ok = x[validcol].to_numpy(dtype=bool)
    p = np.where(ok, x[pcol].to_numpy(dtype=float), 1.0)
    x["Q_BH_trait"] = np.where(ok, adjusted_p(p), np.nan)
    x["Q_BY_trait"] = np.where(ok, adjusted_p(p, "BY"), np.nan)
    x["P_Bonf_trait"] = np.where(ok, np.minimum(p * M_PANEL, 1), np.nan)
    x["nominal"] = ok & (p < ALPHA)
    x["BH_significant"] = ok & (x["Q_BH_trait"] < ALPHA)
    x["Bonf_significant"] = ok & (p < ALPHA / M_PANEL)
    return x

all_results = {}
run_status = []
for trait in TRAITS:
    info = prepared[trait]
    chunks = []
    for c in CHROMS:
        print(f"{trait}: chr{c} / 22", flush=True)
        out = RUN / f"{trait}_results" / f"{trait}_chr{c}"
        cmd = glm_command(GENO / f"BMI_chr{c}", info["pheno"], trait, info["covar"], info["q"], out)
        checked_run(cmd, out)
        z = read_glm(str(out) + f".{trait}.glm.linear")
        x = extract_interaction(z, info["q"], PANEL.loc[PANEL["#CHROM"].eq(str(c))], f"{trait} chr{c}")
        require(x["OBS_CT"].dropna().le(info["N"]).all(), "OBS_CT exceeds prepared cohort")
        x["trait"] = trait
        x["age_center_years"] = info["transform"]["age_center_years"]
        x["genotype_missing_fraction"] = 1 - x["OBS_CT"] / info["N"]
        x["MAF_analysis"] = np.minimum(x["A1_FREQ"], 1-x["A1_FREQ"])
        x["MAC_dosage_approx"] = 2 * x["OBS_CT"] * x["MAF_analysis"]
        x["QC_low_MAC"] = x["MAC_dosage_approx"] < 100
        x["QC_genotype_missing_gt_02"] = x["genotype_missing_fraction"] > 0.02
        x["QC_P_zero"] = x["P"].eq(0) & x["VALID"]
        chunks.append(x)
        run_status.append({"trait": trait, "chromosome": c, "return_code": 0,
                           "n_rows": len(x), "n_valid": int(x["VALID"].sum()),
                           "n_failed": int((~x["VALID"]).sum()), "output": str(out)})
        pd.DataFrame(run_status).to_csv(QC / "chromosome_runs.tsv", sep="\t", index=False)
    x = annotate_multiple_testing(pd.concat(chunks, ignore_index=True))
    # Optional annotation: exact chromosome-position-ID-allele matching, never ID-only deduplication.
    manifest_path = FINAL / "variant_manifest_strict_biallelic.tsv"
    if manifest_path.is_file():
        manifest = read_table(manifest_path)
        require(set(KEY + ["rsid"]).issubset(manifest), "Unexpected manifest schema")
        manifest["#CHROM"] = manifest["#CHROM"].str.replace(r"^chr", "", regex=True)
        manifest["POS"] = pd.to_numeric(manifest["POS"], errors="raise").astype(int)
        ann = manifest[KEY + ["rsid"]].drop_duplicates()
        require(not ann.duplicated(KEY).any(), "Conflicting manifest annotations")
        x = x.merge(ann, on=KEY, how="left", validate="one_to_one")
        save_json(QC / "variant_manifest_source.json", {"path": str(manifest_path), "sha256": sha256(manifest_path)})
    all_results[trait] = x
    x.to_csv(RUN / f"{trait}_linear_age_all.tsv", sep="\t", index=False)
    x.loc[~x["VALID"]].to_csv(QC / f"{trait}_failed_tests.tsv", sep="\t", index=False)

combined = pd.concat([all_results[t] for t in TRAITS], ignore_index=True)
ok = combined["VALID"].to_numpy(dtype=bool)
p_for_family = np.where(ok, combined["P"].to_numpy(dtype=float), 1.0)
combined["Q_BH_all_traits"] = np.where(ok, adjusted_p(p_for_family), np.nan)
combined["P_Bonf_all_traits"] = np.where(ok, np.minimum(p_for_family * len(combined), 1), np.nan)
combined["Bonf_all_traits_significant"] = ok & (p_for_family < ALPHA / len(combined))
combined.to_csv(RUN / "all_traits_linear_age.tsv", sep="\t", index=False)
for trait in TRAITS:
    all_results[trait] = combined.loc[combined["trait"].eq(trait)].copy()
    all_results[trait].to_csv(RUN / f"{trait}_linear_age_all.tsv", sep="\t", index=False)
    all_results[trait].loc[all_results[trait]["VALID"]].sort_values("P").to_csv(
        RUN / f"{trait}_linear_age_valid_sorted.tsv", sep="\t", index=False)
print("Linear fits finished:", RUN)


## 6. QC 汇总、QQ 和面板 Manhattan

QQ 使用有效的真实 P 值；灰带为独立 Uniform P 假设下的逐点 95% order-statistic 区间，LD 下仅作参考。这里是已选择的候选 SNP 面板，Manhattan 不是全基因组扫描；虚线为实际面板的 Bonferroni 阈值，另标出跨三个 trait 的阈值。

`lambda_1df = median[chi2.isf(P,1)] / chi2.ppf(.5,1)`，旧模型使用 df=4 的对应映射。这里用 P 映射到 chi-square 尺度，**不是把 PLINK 的 F 值直接当成 chi-square**。lambda 仅作描述，不做 genomic control 除法，也不证明没有混杂或存在 enrichment。

P=0 保留在结果表中；绘图下限设为 $10^{-300}$ 并统计截断数，不悄悄删除。MAC、missingness 阈值是诊断标记，不改变主要检验族。少量失败 SNP 不会让整个 notebook 丢掉已完成结果；若所有 SNP 都失败，停止。


In [ ]:
def plotting_p(values):
    return np.clip(np.asarray(values, dtype=float), 1e-300, 1.0)

def lambda_from_p(values, df):
    p = np.asarray(values, dtype=float)
    if not len(p):
        return np.nan
    return float(np.median(stats.chi2.isf(p, df)) / stats.chi2.ppf(0.5, df))

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(FIG / f"{stem}.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

def make_plots(df, trait, pcol="P", validcol="VALID", test_df=1, suffix="linear"):
    x = df.loc[df[validcol]].copy()
    require(len(x) > 0, f"{trait}: no valid tests to plot")
    p = np.sort(plotting_p(x[pcol]))
    n = len(p)
    ranks = np.arange(1, n+1)
    expected = -np.log10((ranks - 0.5) / n)
    observed = -np.log10(p)
    lo = -np.log10(np.maximum(stats.beta.ppf(0.975, ranks, n+1-ranks), 1e-300))
    hi = -np.log10(np.maximum(stats.beta.ppf(0.025, ranks, n+1-ranks), 1e-300))
    lam = lambda_from_p(x[pcol], test_df)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.fill_between(expected, lo, hi, color="#dbe4ee", label="95% pointwise band (independent P)")
    ax.scatter(expected, observed, s=16, color="#235f8d", alpha=0.8)
    bound = max(expected.max(), observed.max()) * 1.03
    ax.plot([0,bound], [0,bound], "--", color="grey", linewidth=1)
    ax.set(xlabel="Expected -log10(P)", ylabel="Observed -log10(P)",
           title=f"{trait}: {test_df}-df {suffix} interaction; N variants={n}\nlambda={lam:.3f}")
    ax.legend(fontsize=8)
    save_figure(fig, f"{trait}_{suffix}_QQ")

    x["CHR"] = pd.to_numeric(x["#CHROM"], errors="raise")
    x = x.sort_values(["CHR", "POS"])
    offset, ticks, labels = 0.0, [], []
    fig, ax = plt.subplots(figsize=(13,4.8))
    for c in CHROMS:
        panel_chr = PANEL.loc[PANEL["#CHROM"].eq(str(c))]
        part = x.loc[x["CHR"].eq(c)]
        ax.scatter(part["POS"] + offset, -np.log10(plotting_p(part[pcol])),
                   s=15, color=["#235f8d", "#59a6b1"][c%2], alpha=0.85)
        ticks.append(offset + (panel_chr["POS"].min() + panel_chr["POS"].max()) / 2)
        labels.append(str(c))
        offset += float(panel_chr["POS"].max()) + 5e6
    ax.axhline(-np.log10(ALPHA/M_PANEL), color="#b55d27", linestyle="--", label=f"0.05/{M_PANEL} (per trait)")
    if suffix == "linear":
        ax.axhline(-np.log10(ALPHA/(M_PANEL*len(TRAITS))), color="#773c8b", linestyle=":", label="Across 3 traits")
    ax.set_xticks(ticks, labels)
    ax.set(xlabel="Chromosome (candidate-panel coordinates)", ylabel="-log10(P)",
           title=f"{trait}: {test_df}-df {suffix} interaction — selected variant panel")
    ax.legend(fontsize=8)
    save_figure(fig, f"{trait}_{suffix}_Manhattan")
    return lam

summary_rows = []
for trait in TRAITS:
    x = all_results[trait]
    ok = x["VALID"]
    require(ok.any(), f"{trait}: all tests failed; review logs and QC")
    lam = make_plots(x, trait)
    summary_rows.append({"trait": trait, "N_prepared": prepared[trait]["N"], "N_panel": M_PANEL,
        "N_valid": int(ok.sum()), "N_failed": int((~ok).sum()),
        "N_nominal": int(x["nominal"].sum()), "N_BH_trait": int(x["BH_significant"].sum()),
        "N_Bonf_trait": int(x["Bonf_significant"].sum()),
        "N_Bonf_all_traits": int(x["Bonf_all_traits_significant"].sum()),
        "Bonf_threshold_trait": ALPHA/M_PANEL, "lambda_1df": lam,
        "median_OBS_CT": x.loc[ok,"OBS_CT"].median(), "min_OBS_CT": x.loc[ok,"OBS_CT"].min(),
        "max_OBS_CT": x.loc[ok,"OBS_CT"].max(), "N_low_MAC": int(x["QC_low_MAC"].sum()),
        "N_missing_gt_02": int(x["QC_genotype_missing_gt_02"].sum()),
        "N_P_zero": int(x["QC_P_zero"].sum()),
        "N_P_plot_clipped": int((ok & x["P"].lt(1e-300)).sum())})
    x.groupby(["ERRCODE", "ERRCODE_joint"], dropna=False).size().rename("N").reset_index().to_csv(
        QC / f"{trait}_error_counts.tsv", sep="\t", index=False)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RUN / "linear_trait_summary.tsv", sep="\t", index=False)
display(summary)


## 7. 与旧 B1 4-df 结果比较

只读取旧 `USER_4DF` 结果，不运行新的 spline GWAS。用 chromosome + position + ID + REF + ALT 一对一对齐，不直接按 ID 丢弃重复行。报告旧/新缺失、ERRCODE、A1 方向和 OBS_CT；四 df 的 omnibus P 不依赖 A1 翻转，因此 A1 翻转可比较 P，但不能直接比较 beta 的方向。

下面重算旧模型的 BH / Bonferroni，而不信任旧文件中的布尔列。检查原始日志（若存在）中的 `--parameters` / `--tests`，并验证旧 F/P 的自由度约定。比较使用历史原始 P，另存 `P_residual_df_recomputed_4df`（F(4, N−参数数)）作为核查，绝不静默覆盖旧 P。旧版本 joint P 的 OBS_CT 分母约定与上节相同。**OBS_CT 相同仅说明人数相同，不证明个体身份相同**；旧日志没有输入文件 hash，所以不能自动证明它运行时的 phenotype / covariate 与今天完全相同。当前结果将保留此比较限制。

线性 1-df 与 spline 4-df P 的散点、显著性重叠是描述性比较。不能用 P 值之差检验非线性，也不能因为 linear 不显著而宣布只有非线性效应。正式的“超出线性的非线性”需要在同样本、同基因型缺失模式、同 nuisance 下拟合嵌套模型并进行 3-df 额外交互项检验；这里不计算这个尚未执行的检验。


In [ ]:
def load_old(trait):
    path = B1 / f"{trait}_age_varying_joint_tests.tsv"
    old = read_glm(path)
    require(old["TEST"].eq("USER_4DF").all(), f"{trait}: old table contains non-4df tests")
    require(not old.duplicated(KEY).any(), f"{trait}: duplicate old variant keys")
    old["VALID"] = (old["ERRCODE"].eq(".") & old["P"].between(0,1) &
                    np.isfinite(old["T_OR_F_STAT"]) & old["T_OR_F_STAT"].ge(0))
    # old: intercept + G + q main covariates + 4 interaction columns => q+6 coefficients
    old["DF_RESID_4df"] = old["OBS_CT"] - (prepared[trait]["q"] + 6)
    old["VALID"] &= old["DF_RESID_4df"].gt(0)
    good = old["VALID"] & old["P"].gt(0)
    old["P_residual_df_recomputed"] = stats.f.sf(old["T_OR_F_STAT"], 4, old["DF_RESID_4df"])
    lp_resid = stats.f.logsf(old.loc[good,"T_OR_F_STAT"], 4, old.loc[good,"DF_RESID_4df"])
    lp_obs = stats.f.logsf(old.loc[good,"T_OR_F_STAT"], 4, old.loc[good,"OBS_CT"])
    match_resid = np.isclose(np.log(old.loc[good,"P"]), lp_resid, rtol=4e-4, atol=4e-4)
    match_obs = np.isclose(np.log(old.loc[good,"P"]), lp_obs, rtol=4e-4, atol=4e-4)
    old["F_P_consistent"] = False
    old["joint_P_df_convention"] = "invalid_or_zero"
    old.loc[good,"F_P_consistent"] = match_resid | match_obs
    old.loc[good,"joint_P_df_convention"] = np.where(match_resid & match_obs, "both_within_rounding",
                                                    np.where(match_resid, "residual_df", np.where(match_obs, "OBS_CT_df", "unrecognized")))
    old.loc[old["VALID"] & old["P"].eq(0),"F_P_consistent"] = True  # P=0 retained as underflow
    old["F_P_inconsistent"] = old["VALID"] & ~old["F_P_consistent"]
    old["VALID"] &= old["F_P_consistent"]
    return old

comparisons = {}
comparison_summary = []
old_log_audit = []
for trait in TRAITS:
    old = load_old(trait)
    # Verify historical parameter settings when logs are available.
    q = prepared[trait]["q"]
    for c in CHROMS:
        log = B1 / f"{trait}_results" / f"{trait}_chr{c}.log"
        status = "log_missing"
        if log.is_file():
            txt = log.read_text(errors="replace")
            params = re.search(r"^\s*--parameters\s+([^\r\n]+)", txt, re.M)
            tests = re.search(r"^\s*--tests\s+([^\r\n]+)", txt, re.M)
            matches = (params is not None and tests is not None and
                       params.group(1).strip() == f"1-{q+5}" and tests.group(1).strip() == f"{q+2}-{q+5}")
            status = "parameters_match" if matches else "parameters_mismatch_or_unreadable"
        old_log_audit.append({"trait": trait, "chromosome": c, "status": status, "path": str(log)})
    old = old.rename(columns={c: c + "_4df" for c in old.columns if c not in KEY})
    new = all_results[trait]
    joined = new.merge(old, on=KEY, how="outer", validate="one_to_one", indicator="match_status")
    log_by_chr = {str(r["chromosome"]): r["status"] for r in old_log_audit if r["trait"] == trait}
    joined["old_log_parameter_status"] = joined["#CHROM"].map(log_by_chr)
    joined["same_OBS_CT"] = joined["OBS_CT"].eq(joined["OBS_CT_4df"])
    joined["same_A1"] = joined["A1"].eq(joined["A1_4df"])
    joined["comparable_descriptive"] = (
        joined["match_status"].eq("both") & joined["VALID"].fillna(False).astype(bool) &
        joined["VALID_4df"].fillna(False).astype(bool) & joined["same_OBS_CT"] &
        joined["F_P_consistent_4df"].fillna(False).astype(bool) &
        joined["old_log_parameter_status"].ne("parameters_mismatch_or_unreadable"))
    joined["historical_input_identity_verified"] = False
    joined.to_csv(QC / f"{trait}_old_new_join_audit.tsv", sep="\t", index=False)
    # Full intended panel: absent/failed old tests count conservatively as P=1 for correction.
    old_panel = PANEL.merge(old, on=KEY, how="left", validate="one_to_one")
    old_panel["VALID_4df"] = old_panel["VALID_4df"].fillna(False).astype(bool)
    old_panel = annotate_multiple_testing(old_panel, "P_4df", "VALID_4df")
    correction = old_panel[KEY + ["Q_BH_trait", "P_Bonf_trait", "BH_significant", "Bonf_significant"]].rename(
        columns={k:k+"_4df_recomputed" for k in ("Q_BH_trait", "P_Bonf_trait", "BH_significant", "Bonf_significant")})
    joined = joined.merge(correction, on=KEY, how="left", validate="one_to_one")
    joined.to_csv(RUN / f"{trait}_linear_vs_old_4df.tsv", sep="\t", index=False)
    comparisons[trait] = joined
    comp = joined.loc[joined["comparable_descriptive"]].copy()
    if len(comp) >= 2 and comp["P"].nunique() > 1 and comp["P_4df"].nunique() > 1:
        rho = float(stats.spearmanr(comp["P"], comp["P_4df"]).statistic)
    else:
        rho = np.nan
    old_lam = lambda_from_p(old_panel.loc[old_panel["VALID_4df"],"P_4df"], 4)
    comparison_summary.append({"trait": trait, "N_new": len(new), "N_old": len(old),
        "N_match_keys": int(joined["match_status"].eq("both").sum()),
        "N_new_only": int(joined["match_status"].eq("left_only").sum()),
        "N_old_only": int(joined["match_status"].eq("right_only").sum()),
        "N_OBS_CT_mismatch": int((joined["match_status"].eq("both") & ~joined["same_OBS_CT"]).sum()),
        "N_A1_different": int((joined["match_status"].eq("both") & ~joined["same_A1"]).sum()),
        "N_old_F_P_inconsistent": int(old["F_P_inconsistent_4df"].sum()),
        "N_comparable_descriptive": len(comp), "P_rank_correlation": rho,
        "lambda_old_4df": old_lam,
        "N_old_BH_trait": int(old_panel["BH_significant"].sum()),
        "N_old_Bonf_trait": int(old_panel["Bonf_significant"].sum()),
        "N_Bonf_overlap_comparable": int((comp["Bonf_significant"] & comp["Bonf_significant_4df_recomputed"].astype(bool)).sum())})
    if len(comp):
        fig, ax = plt.subplots(figsize=(6,6))
        ax.scatter(-np.log10(plotting_p(comp["P_4df"])), -np.log10(plotting_p(comp["P"])), s=16, alpha=0.7)
        threshold = -np.log10(ALPHA/M_PANEL)
        ax.axvline(threshold, linestyle="--", color="grey")
        ax.axhline(threshold, linestyle="--", color="grey")
        ax.set(xlabel="Old 4-df interaction -log10(P)", ylabel="Linear 1-df interaction -log10(P)",
               title=f"{trait}: descriptive comparison; matched N variants={len(comp)}")
        save_figure(fig, f"{trait}_linear_vs_4df")
        for metric, old_metric, label in [
            ("BH_significant", "BH_significant_4df_recomputed", "BH"),
            ("Bonf_significant", "Bonf_significant_4df_recomputed", "Bonf")]:
            overlap = pd.crosstab(comp[metric].astype(bool), comp[old_metric].astype(bool),
                                 rownames=["linear_significant"], colnames=["old_4df_significant"], dropna=False)
            overlap = overlap.reindex(index=[False,True], columns=[False,True], fill_value=0)
            overlap.to_csv(RUN / f"{trait}_{label}_overlap.tsv", sep="\t")
    if old_panel["VALID_4df"].any():
        make_plots(old_panel, trait, pcol="P_4df", validcol="VALID_4df", test_df=4, suffix="old_4df")
pd.DataFrame(old_log_audit).to_csv(QC / "old_log_parameters.tsv", sep="\t", index=False)
comparison_summary = pd.DataFrame(comparison_summary)
comparison_summary.to_csv(RUN / "linear_vs_4df_summary.tsv", sep="\t", index=False)
display(comparison_summary)


## 8. 运行完成与结果解释

输出包含每个 SNP 的效应、每年/每十年标准误、95% CI、t、F、P、A1、OBS_CT、ERRCODE、BH/BY/Bonferroni、面板和三 trait 校正；各染色体原始结果和命令日志也保留。`RUN_COMPLETE.json` 只有在所有分析、审计和图完成后才写出。失败或异常用 QC 文件定位，不能只看显著结果。

解读时遵循以下边界：

- 线性检验比 4-df omnibus 更有力，只在效应形状主要沿线性年龄方向时可能成立；实际是否如此由运行后的结果决定。
- 单个正/负 `BETA_per_10y` 表示横断面数据中 A1 effect 的年龄斜率，不直接等同于同一个人的纵向衰老效应或因果效应。
- 普通 OLS 不处理亲缘相关、异方差、未控制的 ancestry / batch / selection effects；sex/QPC 调整不能自动保证这些问题已解决。正式结论应审查上游无关个体选择、残差异方差、年龄分层 genotype count 与缺失模式；必要时用 robust / mixed-model 方法复核信号。
- 这是候选面板的单 SNP 检验与诊断，不是已完成的 enrichment 检验。nominal hit 数量、QQ 尾部偏离和 lambda 不能单独证明 enrichment，尤其在 LD、原 GWAS 选择及可能的样本重叠存在时。后续 enrichment 需要明确 null、匹配背景、LD 处理和选择过程。
- 若同时把 linear 与旧 spline 两种模型都当作新发现来源，还需预先定义跨模型的多重检验策略；这里的跨 trait 校正只覆盖本次 linear family。

可直接查看 `linear_trait_summary.tsv`、`linear_vs_4df_summary.tsv`、`qc/cohorts.tsv`、`qc/*failed_tests.tsv` 和 `figures/`。不要将 controlled 数据或个人级输入导出 workspace；本 notebook 只在缺少 PLINK2 时下载公开软件包，不下载、同步或上传 controlled data。


In [ ]:
completion = {"status": "complete", "run": str(RUN), "plink_version": PLINK_VERSION,
              "finished_utc": datetime.now(timezone.utc).isoformat(),
              "traits": list(TRAITS), "chromosomes": list(CHROMS), "panel_variants": M_PANEL,
              "linear_summary": summary.to_dict(orient="records"),
              "comparison_summary": comparison_summary.to_dict(orient="records"),
              "historical_input_identity_verified": False}
save_json(RUN / "RUN_COMPLETE.json", completion)
print("Completed. All outputs remain in:", RUN)
display(summary)
display(comparison_summary)


## 验证与参考

交付检查：JSON / nbformat 结构、所有 code cell 语法、相同日期 PLINK v2.0.0-a.7.8 的合成测试、独立 OLS 对照，以及模拟 workspace 的全流程运行（3 个 phenotype × 22 条染色体、19 个 nuisance 协变量；含缺失基因型、低频和单态 variant，生成并检查 15 张图及全部结果表）。未在用户的真实 Verily Linux/Spark driver 或 controlled cohort 上执行。运行本 notebook 时会再次用目标环境的 PLINK 做合成测试，不通过就不继续真实分析。

- 输入来源：上传的 `age_varying(1).ipynb`，以及原对话中的 B1 背景。历史 N 与结果不作为新结果写入输出表。
- [PLINK2 association / parameters / tests](https://www.cog-genomics.org/plink/2.0/assoc#glm)
- [PLINK2 input / phenotype missing-value handling](https://www.cog-genomics.org/plink/2.0/input#pheno)
- [PLINK2 file formats](https://www.cog-genomics.org/plink/2.0/formats)

设计保留了原 notebook 的 phenotype 原始单位、样本、样条空间和 sex/QPC；任何无法核对的变化都会报错或写入 QC，而不推断未检验的结果。

本次 v2 修订只增加软件发现与缺失安装流程；已检查现有程序复用、持久目录回退、安装命令和未完成安装目录保护。安装分支采用模拟测试，尚未在用户实际 Workbench 联网安装。统计分析代码与已完成全流程合成验证的版本相同。
